In [ ]:
import pandas as pd
import numpy as np
import re

# ============================================================
# Load Kaggle YouTube Comments Dataset
# ============================================================
df = pd.read_csv("/content/YoutubeCommentsDataSet.csv")

# Check available columns
print(df.columns)
print(df.head())

# ============================================================
# Select the comment column
# ============================================================
# Change this if your dataset uses another column name
COMMENT_COLUMN = "Comment"

random_comments = df[[COMMENT_COLUMN]].copy()
random_comments.columns = ["comment"]

# ============================================================
# Remove NaN values
# ============================================================
random_comments = random_comments.dropna()

# Convert to string
random_comments["comment"] = random_comments["comment"].astype(str)

# ============================================================
# Basic cleaning
# ============================================================
def clean_text(text):
    text = text.strip()
    text = re.sub(r"http\\S+|www\\S+", "", text)
    text = re.sub(r"\\s+", " ", text)
    return text

random_comments["comment"] = random_comments["comment"].apply(clean_text)

# Remove empty comments
random_comments = random_comments[
    random_comments["comment"].str.len() > 3
]

# ============================================================
# Climate-related keywords (English + Hindi + Bengali)
# ============================================================
keywords = [
    # English
    "climate", "global warming", "greenhouse", "carbon", "co2",
    "environment", "pollution", "renewable", "fossil", "temperature",
    "climate change", "warming", "emission", "emissions",
    "net zero", "ozone", "sea level", "heatwave", "wildfire",
    # Hindi
    "जलवायु", "ग्लोबल वार्मिंग", "पर्यावरण", "कार्बन",
    "तापमान", "प्रदूषण", "उत्सर्जन",
    # Bengali
    "জলবায়ু", "গ্লোবাল ওয়ার্মিং", "পরিবেশ", "কার্বন",
    "তাপমাত্রা", "দূষণ", "নিঃসরণ"
]

# Create regex pattern
pattern = "|".join([re.escape(k) for k in keywords])

# ============================================================
# Remove climate-related comments
# ============================================================
filtered_comments = random_comments[
    ~random_comments["comment"]
      .str.lower()
      .str.contains(pattern, regex=True, na=False)
]

# ============================================================
# Remove duplicates
# ============================================================
filtered_comments = filtered_comments.drop_duplicates()

# ============================================================
# Sample fixed number of comments
# ============================================================
SAMPLE_SIZE = 10000

if len(filtered_comments) > SAMPLE_SIZE:
    filtered_comments = filtered_comments.sample(
        n=SAMPLE_SIZE,
        random_state=42
    )

# ============================================================
# Label all as NONE
# ============================================================
filtered_comments["label"] = "NONE"

# Rename column for training
filtered_comments.columns = ["sentence", "label"]

# ============================================================
# Save final dataset
# ============================================================
SAVE_PATH = "/content/non_climate_none.csv"

filtered_comments.to_csv(SAVE_PATH, index=False)

# ============================================================
# Display results
# ============================================================
print("Total comments after filtering :", len(filtered_comments))
print(filtered_comments.head(10))

print(f"Saved to: {SAVE_PATH}")

Index(['Comment', 'Sentiment'], dtype='object')
                                             Comment Sentiment
0  lets not forget that apple pay in 2014 require...   neutral
1  here in nz 50 of retailers don’t even have con...  negative
2  i will forever acknowledge this channel with t...  positive
3  whenever i go to a place that doesn’t take app...  negative
4  apple pay is so convenient secure and easy to ...  positive
Total comments after filtering : 10000
                                                sentence label
15537  im so hyped for starfield but i feel like i ca...  NONE
16133  eres un ídolo amdreleasemovieuno siempre en mi...  NONE
5021   been using my m2 mba for 2 weeks now the best ...  NONE
10575                     dinner for one can be relaxing  NONE
13952  well i disagree with the answer to question 27...  NONE
8038   this episode was truly hilarious and just like...  NONE
11468  india is a country of a billion people but if ...  NONE
7594      music doesn’t give me